# AKE Lab 1 · MLP für MNIST — Starter-Notebook

**Angewandte KI & KI-Engineering · Block A – Vertiefung Deep Learning · Woche 1**
Zugehöriges Skript: *Vom Perzeptron zum Deep Network*, Abschnitt 9 · Aufgabenblatt: `AKE_Lab01_Aufgabenblatt.pdf`

Arbeiten Sie die Zellen der Reihe nach ab. Stellen mit `TODO` bzw. `...` sind von Ihnen zu ergänzen. Nach jedem Experiment folgt eine Markdown-Zelle **Interpretation**. Dort werden 3–6 Sätze erwartet, die Ihr Ergebnis mit dem Skript verknüpfen (Gleichungs- oder Tabellennummer angeben).

**Rechenbudget (Referenzhardware 4 Kerne / 8 GB, keine GPU):** reine Rechenzeit ≈ 8–12 Min. Alle
Experimente schreiben ihre Trainingslogs als JSON nach `results/`. Bricht ein eigener Lauf ab, können Sie die
mitgelieferten Referenzlogs aus `results_referenz/` laden und die Analyse trotzdem bearbeiten.

**Tipps für alle mit wenig Programmiererfahrung**
- Zelle ausführen: *Shift + Enter*. Zellen bauen aufeinander auf – nach einem Kernel-Neustart alles von oben erneut ausführen.
- Alles hinter `#` ist ein Kommentar und wird nicht ausgeführt. `...` ist ein Platzhalter: Ersetzen Sie ihn durch Ihren Code.
- Zeilen mit `Hinweis:` unter einem `TODO` geben Tipps zur Python-Schreibweise, nicht zur Lösung.
- Fehlermeldungen von **unten** lesen: Die letzte Zeile nennt den Fehler, darüber steht die betroffene Codezeile.
- Hilfe zu einer Funktion: `help(torch.optim.SGD)` oder `torch.optim.SGD?` in einer eigenen Zelle.

## 0 · Setup und Daten

Importe, Pfade, MNIST-Splits (54 000 / 6 000 / 10 000) wie in Skript Listing 3. Der Testsplit wird **ausschließlich** für die finale Bewertung je Aufgabe benutzt – nie zur Modellauswahl (Abschn. 6.1).

In [ ]:
# --- Bibliotheken importieren -------------------------------------------------
# "import" macht fremden Code (Bibliotheken) in diesem Notebook verfuegbar.
import os, time, json, math, copy   # Standardbibliothek: Zeitmessung, JSON-Dateien, Mathe, Kopien von Objekten
from pathlib import Path            # Path: komfortabler Umgang mit Datei- und Ordnerpfaden

import numpy as np                  # NumPy: Rechnen mit Arrays (Vektoren/Matrizen); Kurzname "np" ist Konvention
import torch, torch.nn as nn        # PyTorch: Tensoren + automatisches Ableiten; nn = Bausteine fuer neuronale Netze
from torchvision import datasets, transforms                    # fertige Datensaetze (MNIST) und Bildvorverarbeitung
from torch.utils.data import DataLoader, Subset, random_split   # Daten in Batches liefern, Teilmengen bilden, aufteilen
import matplotlib.pyplot as plt     # Plotten; Kurzname "plt" ist Konvention

# Zufallsgeneratoren fest initialisieren ("Seed"). Gewichtsinitialisierung und Datenmischung liefern
# dann bei jedem Neustart dieselben Zufallszahlen -> Ergebnisse sind reproduzierbar.
torch.manual_seed(0); np.random.seed(0)

DATA_ROOT = "../data"               # relativer Pfad: ".." = eine Ordnerebene ueber diesem Notebook
RESULTS   = Path("./results"); RESULTS.mkdir(exist_ok=True)   # Trainingslogs je Experiment (JSON)
# exist_ok=True: kein Fehler, falls der Ordner results/ schon existiert

# Kurzer Umgebungscheck: PyTorch-Version, Anzahl CPU-Threads, ob eine GPU (CUDA) verfuegbar ist
print("torch", torch.__version__, "| Threads:", torch.get_num_threads(), "| CUDA:", torch.cuda.is_available())

In [ ]:
# --- Vorverarbeitung ----------------------------------------------------------
# Jedes Bild durchlaeuft diese Kette (Compose = "nacheinander ausfuehren"):
#   1. ToTensor():  Bild mit Pixeln 0..255 -> Tensor der Form (1, 28, 28) mit Werten 0..1
#                   (1 = ein Farbkanal, da Graustufen; 28 x 28 Pixel)
#   2. Normalize(): rechnet (x - 0.1307) / 0.3081 -> Pixel haben danach ca. Mittelwert 0 und Std 1,
#                   das erleichtert dem Optimierer die Arbeit. Das Komma in (0.1307,) macht daraus
#                   ein Tupel mit einem Eintrag (ein Wert je Farbkanal).
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.1307,), (0.3081,))])   # Mittelwert/Std von MNIST

# --- Datensaetze laden --------------------------------------------------------
# train=True -> 60 000 Trainingsbilder, train=False -> 10 000 Testbilder.
# download=True laedt nur herunter, wenn die Daten unter DATA_ROOT noch fehlen.
full = datasets.MNIST(DATA_ROOT, train=True,  download=True, transform=tf)
test = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=tf)
# Die 60 000 Trainingsbilder zufaellig aufteilen: 54 000 zum Trainieren, 6 000 zur Validierung
# (Modellauswahl, Early Stopping). Der Generator mit festem Seed sorgt dafuer, dass die Aufteilung
# bei jedem Lauf identisch ist. Die Unterstriche in 54_000 dienen nur der Lesbarkeit (= 54000).
train, val = random_split(full, [54_000, 6_000], generator=torch.Generator().manual_seed(0))

def make_loaders(train_ds=train, batch_size=64):
    """Train-Loader wird gemischt, Val/Test nicht. Val/Test sind immer die vollen Splits."""
    # Ein DataLoader liefert die Daten nicht einzeln, sondern in Paketen ("Batches"): Jeder
    # Schleifendurchlauf ergibt x mit Form (64, 1, 28, 28) und y (die richtigen Ziffern) mit Form (64,).
    # shuffle=True mischt die Trainingsdaten vor jeder Epoche neu, damit das Netz keine Reihenfolge lernt.
    # Val/Test werden nur ausgewertet -> groessere Batches (256) sind schneller, Mischen ist unnoetig.
    # Rueckgabe ist ein Dictionary; Zugriff spaeter z. B. mit dl["train"].
    return {"train": DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            "val":   DataLoader(val,      batch_size=256),
            "test":  DataLoader(test,     batch_size=256)}

dl = make_loaders()   # Standard-Loader: volle 54 000 Trainingsbilder, Batchgroesse 64
# f"..." ist ein f-String: Ausdruecke in {} werden ausgewertet und in den Text eingesetzt.
# 54 000 / 64 = 843,75 -> 844 Batches pro Epoche (der letzte Batch ist kleiner).
print(f"Train {len(train)} | Val {len(val)} | Test {len(test)} | Batches/Epoche: {len(dl['train'])}")

# --- Plausibilitaetscheck: erste 8 Bilder mit ihrem Label anzeigen -------------
fig, ax = plt.subplots(1, 8, figsize=(8, 1.4))   # 1 Zeile mit 8 Teilbildern; figsize in Zoll
for i in range(8):                               # i = 0, 1, ..., 7
    x, y = full[i]                               # ein Datensatzeintrag = (Bildtensor, Label)
    # x[0] waehlt den einzigen Farbkanal -> 28x28-Matrix; "gray_r" = Graustufen, invertiert (schwarz auf weiss)
    ax[i].imshow(x[0], cmap="gray_r"); ax[i].set_title(y, fontsize=9); ax[i].axis("off")
plt.show()

### Modell, Trainings- und Evaluationsschleife

`MLP` verallgemeinert Listing 1 (variable Tiefe, Breite, Aktivierung, optional Dropout). `run_epoch`/`fit` sind die Referenzschleife aus Listing 3, ergänzt um Logging, Early Stopping und JSON-Export. Diese Schleife bleibt im Rest des Semesters praktisch unverändert.

In [ ]:
class MLP(nn.Module):
    """MLP 784 -> hidden[0] -> ... -> n_classes. Ausgabe sind Logits (Softmax steckt im Loss).
    act: Aktivierungsklasse (nn.ReLU, nn.Sigmoid, ...); dropout: p nach jeder Aktivierung."""
    # Ein PyTorch-Modell ist eine Klasse, die von nn.Module erbt. Zwei Methoden sind zentral:
    #   __init__: laeuft einmal beim Anlegen (model = MLP()) und baut die Schichten auf
    #   forward:  beschreibt, wie eine Eingabe x durch die Schichten zur Ausgabe wird
    # "self" ist das Objekt selbst; Attribute wie self.net bleiben so nach dem Aufruf erhalten.
    def __init__(self, d_in=784, hidden=(128, 128), n_classes=10, act=nn.ReLU, dropout=0.0, init="default"):
        # Parameter mit "=" haben Standardwerte und koennen beim Aufruf weggelassen werden:
        #   d_in=784          28 * 28 Pixel pro Bild
        #   hidden=(128, 128) Breite jeder verdeckten Schicht; Anzahl Eintraege = Anzahl verdeckter Schichten
        #   act=nn.ReLU       die Klasse selbst (noch kein Objekt!) -- act() erzeugt je Schicht eine eigene Aktivierung
        super().__init__()   # Pflichtaufruf: richtet die Grundfunktionen von nn.Module ein
        # layers sammelt alle Bausteine in der Reihenfolge, in der die Daten sie durchlaufen.
        # nn.Flatten() macht aus jedem Bild (1, 28, 28) einen Vektor mit 784 Eintraegen.
        # d merkt sich die Ausgabebreite des zuletzt angehaengten Bausteins (anfangs 784).
        layers, d = [nn.Flatten()], d_in
        for h in hidden:     # einmal pro verdeckter Schicht; h = deren Breite
            # TODO: nn.Linear(d, h) und act() anhaengen; bei dropout > 0 zusaetzlich nn.Dropout(dropout)
            #       (Reihenfolge laut Skript: Linear -> Aktivierung -> Dropout)
            #       Hinweis: an eine Liste haengt man mit layers.append(...) ein Element an.
            ...
            d = h            # Ausgabebreite dieser Schicht = Eingabebreite der naechsten
        # TODO: Ausgabeschicht nn.Linear(d, n_classes) anhaengen, OHNE Softmax
        ...
        # nn.Sequential fuehrt die Bausteine spaeter in genau dieser Reihenfolge aus.
        # Der Stern in *layers "entpackt" die Liste: Sequential(a, b, c) statt Sequential([a, b, c]).
        self.net = nn.Sequential(*layers)
        self.reset_parameters(init)   # Gewichte ggf. nach einer anderen Regel neu initialisieren

    def reset_parameters(self, init):
        """init: "default" (PyTorch: kaiming_uniform mit a=sqrt(5), d.h. Var = 1/(3 n_in)),
        "glorot" (Var = 2/(n_in+n_out), Gl. 24) oder "he" (Var = 2/n_in, Gl. 25). Biases = 0."""
        if init == "default":
            return                         # PyTorch hat beim Anlegen bereits initialisiert -> nichts zu tun
        for m in self.net:                 # alle Bausteine durchgehen ...
            if isinstance(m, nn.Linear):   # ... aber nur Linear-Schichten haben Gewichte
                # Kompakte Schreibweise: Das Dictionary ordnet jedem Namen eine Funktion zu, [init] waehlt
                # die passende aus und (m.weight) ruft sie auf. Ausgeschrieben bedeutet das:
                #   if init == "glorot": nn.init.xavier_normal_(m.weight)
                #   if init == "he":     nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                # Unterstrich am Namensende (xavier_normal_) heisst in PyTorch: aendert den Tensor direkt ("in place").
                {"glorot": nn.init.xavier_normal_,
                 "he": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu")}[init](m.weight)
                nn.init.zeros_(m.bias)     # alle Biases auf 0

    def forward(self, x):
        # Wird automatisch aufgerufen, wenn man model(x) schreibt.
        # x: Batch von Bildern, Form (B, 1, 28, 28) -> Rueckgabe: Logits, Form (B, 10), ein Wert je Ziffer
        return self.net(x)

# --- Selbsttest der Architektur -----------------------------------------------
model = MLP()   # Standardnetz 784-128-128-10
# model.parameters() liefert alle lernbaren Tensoren (Gewichte und Biases); numel() zaehlt deren Eintraege.
# Erwartung: (784*128 + 128) + (128*128 + 128) + (128*10 + 10) = 100 480 + 16 512 + 1 290 = 118 282
n_params = sum(p.numel() for p in model.parameters())
print(model.net)   # zeigt die Schichtfolge -- gut zum Pruefen Ihrer TODOs
print("Parameter:", n_params)
# assert bricht mit der Meldung ab, wenn die Bedingung nicht erfuellt ist
assert n_params == 118_282, "Parameterzahl weicht von Skript Bsp. 3.5 ab -- Architektur pruefen"

In [ ]:
# Verlustfunktion fuer Klassifikation. CrossEntropyLoss wendet intern Softmax an und bewertet, wie viel
# Wahrscheinlichkeit auf die richtige Klasse entfaellt. Deshalb bekommt sie die rohen Netzausgaben (Logits);
# ein zusaetzliches Softmax im Modell waere doppelt. Die Labels y sind einfach Ziffern 0..9 (kein One-Hot noetig).
lossf = nn.CrossEntropyLoss()   # erwartet Logits, kein Softmax davor!

def run_epoch(model, loader, opt=None):
    """Eine Epoche. opt=None => reine Evaluation (kein Gradient, eval()-Modus)."""
    # Dieselbe Funktion dient fuer Training UND Auswertung:
    #   mit Optimierer opt -> Training: Gradienten berechnen und Gewichte anpassen
    #   ohne Optimierer    -> Evaluation: nur vorhersagen und messen
    train_mode = opt is not None   # True, wenn ein Optimierer uebergeben wurde
    # train(True/False) schaltet Bausteine um, die sich im Training anders verhalten (hier: Dropout)
    model.train(train_mode)
    tot_loss, correct, n = 0.0, 0, 0   # laufende Summen: Verlust, richtige Vorhersagen, Anzahl Bilder
    # Gradienten nur im Training mitrechnen -- bei der Auswertung spart das Zeit und Speicher
    with torch.set_grad_enabled(train_mode):
        for x, y in loader:            # ein Batch pro Durchlauf: x = Bilder, y = richtige Ziffern
            logits = model(x)          # Vorwaertsdurchlauf: Form (B, 10)
            loss = lossf(logits, y)    # mittlerer Verlust ueber den Batch (ein einzelner Wert)
            if train_mode:
                # Die drei Standardschritte jedes Trainingsschritts:
                #   opt.zero_grad()  alte Gradienten loeschen (PyTorch wuerde sie sonst aufaddieren)
                #   loss.backward()  Backpropagation: Gradient des Verlusts nach jedem Gewicht berechnen
                #   opt.step()       Gewichte einen Schritt entgegen dem Gradienten verschieben
                opt.zero_grad(); loss.backward(); opt.step()
            # .item() macht aus einem Tensor mit einem Wert eine normale Python-Zahl.
            # loss ist ein Batch-Mittelwert -> mal Batchgroesse, damit am Ende korrekt ueber alle Bilder
            # gemittelt wird (der letzte Batch ist meist kleiner).
            tot_loss += loss.item() * len(y)
            # argmax(1): Position des groessten Logits je Zeile = vorhergesagte Ziffer.
            # (... == y) ergibt True/False je Bild; .sum() zaehlt die Treffer.
            correct  += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return tot_loss / n, correct / n   # Tupel (mittlerer Verlust, Genauigkeit zwischen 0 und 1)

def fit(model, opt, loaders, epochs, name=None, patience=None, verbose=True):
    """Trainiert `epochs` Epochen, loggt Train-/Val-Verlust und -Genauigkeit.
    patience=k: Early Stopping — Abbruch nach k Epochen ohne Verbesserung des Val-Verlusts,
    beste Parameter werden wiederhergestellt. Rueckgabe: history-Dict (auch als JSON in results/)."""
    # history: je Kennzahl eine Liste, an die pro Epoche ein Wert angehaengt wird
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
    # Merker fuer Early Stopping: bester Val-Verlust bisher, zugehoerige Gewichte, Epochen ohne Verbesserung.
    # float("inf") = unendlich, damit die erste Epoche auf jeden Fall als Verbesserung zaehlt.
    best_loss, best_state, bad = float("inf"), None, 0
    for ep in range(1, epochs + 1):    # Epochen 1..epochs (range schliesst das Ende aus, daher + 1)
        t0 = time.time()               # Startzeit fuer die Zeitmessung
        tr = run_epoch(model, loaders["train"], opt)   # eine Epoche trainieren -> (loss, acc) auf Trainingsdaten
        va = run_epoch(model, loaders["val"])          # danach auf Validierungsdaten auswerten (ohne opt: kein Training)
        # (*tr, *va) setzt beide Tupel zu (train_loss, train_acc, val_loss, val_acc) zusammen; zip paart
        # jeden Wert mit seinem Namen, sodass er an die passende Liste in hist angehaengt wird.
        for k, v in zip(["train_loss", "train_acc", "val_loss", "val_acc"], (*tr, *va)):
            hist[k].append(v)
        hist["epoch_time"].append(time.time() - t0)
        if verbose:
            # Formatangaben: {ep:3d} Ganzzahl, 3 Zeichen breit | {x:.4f} 4 Nachkommastellen | {x:.2%} als Prozent
            # hist['epoch_time'][-1]: Index -1 = letztes Element einer Liste
            print(f"  Epoche {ep:3d}: train loss {tr[0]:.4f} acc {tr[1]:.2%} | "
                  f"val loss {va[0]:.4f} acc {va[1]:.2%} | {hist['epoch_time'][-1]:.1f}s")
        if not math.isfinite(va[0]):   # NaN oder unendlich: Training ist divergiert, Weitermachen sinnlos
            print("  -> Verlust nicht endlich, Abbruch."); break
        if va[0] < best_loss:          # neuer Bestwert auf den Validierungsdaten?
            best_loss, bad = va[0], 0
            # Gewichte sichern. deepcopy ist noetig, weil state_dict() nur auf die aktuellen Tensoren
            # verweist -- und die veraendern sich im weiteren Training noch.
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1                   # eine weitere Epoche ohne Verbesserung
            if patience is not None and bad >= patience:
                print(f"  -> Early Stopping nach Epoche {ep} (bestes Val-Loss {best_loss:.4f} "
                      f"in Epoche {ep - bad}).")
                break                  # Trainingsschleife vorzeitig verlassen
    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)          # Parameterstand am Val-Minimum behalten
    # np.argmin liefert die Position (ab 0) des kleinsten Val-Verlusts; + 1 ergibt die Epochennummer
    hist["best_epoch"] = int(np.argmin(hist["val_loss"])) + 1
    if name:                           # nur speichern, wenn ein Name uebergeben wurde
        # Path-Objekte lassen sich mit "/" verketten -> Datei results/<name>.json; indent=1 macht sie lesbar
        (RESULTS / f"{name}.json").write_text(json.dumps(hist, indent=1))
    return hist

def plot_history(hists, title="", metric="loss", ax=None, logy=False):
    """hists: {label: history}. Zeichnet Train (durchgezogen) und Val (gestrichelt)."""
    # Beispiele:  plot_history({"Baseline": h_base}, title="Aufgabe 1")
    #             plot_history({"ReLU": h1, "Sigmoid": h2}, metric="acc")   # Genauigkeit statt Verlust
    if ax is None:                     # kein Zeichenbereich uebergeben -> neue Abbildung anlegen
        _, ax = plt.subplots(figsize=(6, 3.6))   # "_" = Rueckgabewert, der nicht gebraucht wird (die Figure)
    for i, (label, h) in enumerate(hists.items()):   # enumerate liefert zusaetzlich eine laufende Nummer i
        c = f"C{i}"                    # Matplotlib-Standardfarben C0, C1, ...: train und val einer Variante gleich gefaerbt
        # h[f"train_{metric}"] ist z. B. h["train_loss"]; x-Achse = Epochennummern 1..n
        ax.plot(range(1, len(h[f"train_{metric}"]) + 1), h[f"train_{metric}"], c, label=f"{label} (train)")
        ax.plot(range(1, len(h[f"val_{metric}"]) + 1),   h[f"val_{metric}"],   c + "--", label=f"{label} (val)")  # "--" = gestrichelt
    ax.set_xlabel("Epoche"); ax.set_ylabel(metric); ax.set_title(title); ax.grid(alpha=.3)
    if logy: ax.set_yscale("log")      # logarithmische y-Achse, falls gewuenscht
    ax.legend(fontsize=7)
    return ax                          # Rueckgabe erlaubt weitere Anpassungen, z. B. ax.axhline(...)

## Aufgabe 1 · Baseline

MLP 784–128–128–10 (ReLU), Adam $\alpha=10^{-3}$, Batch 64, 5 Epochen auf den vollen 54 000 Trainingsbildern. Erwartung: ≈ 97–98 % Testgenauigkeit. Vorher der erste Schritt des Debug-Protokolls (Abschn. 8): Verlust beim ersten Batch ≈ ln 10.

In [ ]:
torch.manual_seed(0)                                  # gleicher Startpunkt -> vergleichbare Ergebnisse
model = MLP(hidden=(128, 128))                        # frisches, untrainiertes Netz
# Adam-Optimierer: bekommt die zu lernenden Parameter des Modells und die Lernrate lr (alpha im Skript)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)   # 1e-3 = 0,001

# Debug-Protokoll Schritt 1 (Skript Abschn. 8): Verlust vor dem Training ~ ln(10)?
# Begruendung: Ein untrainiertes Netz verteilt seine Wahrscheinlichkeit ungefaehr gleichmaessig (1/10 je Ziffer).
# Cross-Entropy ist dann -ln(1/10) = ln 10 ~ 2,303. Deutlich andere Werte deuten auf einen Fehler hin.
x0, y0 = next(iter(dl["train"]))   # iter() + next(): genau einen Batch aus dem Loader holen
print(f"Verlust vor Training: {lossf(model(x0), y0).item():.3f}  (erwartet ~ ln 10 = {math.log(10):.3f})")

# TODO: 5 Epochen trainieren (fit), anschliessend auf dl["test"] evaluieren (run_epoch ohne opt)
#   Hinweise: fit gibt das history-Dictionary zurueck (mit name=..., z. B. "a1_baseline", landet es in results/).
#             run_epoch gibt ein Tupel (loss, acc) zurueck, das sich direkt auf zwei Variablen verteilen laesst.
h_base = ...
test_loss, test_acc = ...
print(f"Test: loss {test_loss:.4f} | acc {test_acc:.2%}")

# TODO: Trainings- und Validierungsverlust je Epoche plotten (plot_history)
#   Hinweis: plot_history erwartet ein Dictionary {Bezeichnung: history}, siehe Beispiele in der Funktion.


### Interpretation Aufgabe 1

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 2 · Architektur und Vanishing Gradients

10-schichtiges Netz (10 × 100 Einheiten), einmal mit Sigmoid, einmal mit ReLU, SGD $\epsilon = 0{,}1$. Nach dem ersten Batch die Gradientennorm der ersten Schicht messen (`model.net[1].weight.grad.norm()`) und mit Gl. (29) erklären. Anschließend prüfen, ob die Netze überhaupt lernen.

In [ ]:
def grad_norms_first_batch(model, lr=0.1):
    """Vorwaerts- und Rueckwaertsdurchlauf auf dem ersten Batch (ohne Parameter-Update).
    Rueckgabe: (Loss, ||dL/dz_l|| je Aktivierung -- der Adjungierte aus Gl. 29, wie Abb. 17 --,
    ||dL/dW_l|| je Linear-Schicht)."""
    # Idee: Wir messen, wie gross der Gradient in jeder Schicht ist. Wird er zur Eingabe hin immer
    # kleiner, bekommen die vorderen Schichten kaum noch ein Lernsignal (Vanishing Gradients).
    acts = []                            # hier landen die Ausgaben jeder Aktivierungsschicht
    def keep(module, inp, out):          # Forward-Hook: Aktivierung merken und ihren Gradienten behalten
        # Ein "Hook" ist eine Funktion, die PyTorch bei jedem Vorwaertsdurchlauf eines Bausteins automatisch
        # aufruft. Normalerweise speichert PyTorch Gradienten nur fuer Gewichte; retain_grad() sorgt dafuer,
        # dass nach backward() auch out.grad (= dL/dz dieser Zwischenschicht) verfuegbar ist.
        out.retain_grad(); acts.append(out)
    # Hook an jede Aktivierungsschicht haengen (alles ausser Linear und Flatten, also ReLU bzw. Sigmoid).
    # register_forward_hook gibt einen "Griff" zurueck, mit dem sich der Hook spaeter wieder entfernen laesst.
    hooks = [m.register_forward_hook(keep) for m in model.net if not isinstance(m, (nn.Linear, nn.Flatten))]
    x, y = next(iter(dl["train"]))       # ein Batch aus dem Trainings-Loader
    model.zero_grad()                    # eventuell vorhandene alte Gradienten loeschen
    # TODO: Vorwaertsdurchlauf, Loss, backward()
    #   Hinweis: Die Variable muss loss heissen -- sie wird in der return-Zeile verwendet.
    ...
    for h in hooks: h.remove()           # Hooks entfernen, damit sie spaetere Laeufe nicht beeinflussen
    # .norm() = euklidische Laenge des Gradienten-Tensors (Wurzel aus der Quadratsumme) -- ein Mass fuer seine Groesse
    z_norms = [a.grad.norm().item() for a in acts]                      # Adjungierte je Aktivierung
    # TODO: Liste der Gradientennormen ||dL/dW_l|| aller nn.Linear-Schichten in model.net
    #       (Tipp: m.weight.grad.norm() fuer m in model.net, falls isinstance(m, nn.Linear))
    #       Vorlage fuer die Schreibweise ("List Comprehension"): siehe z_norms und hooks oben
    w_norms = ...
    return loss.item(), z_norms, w_norms

deep = {}                                # Ergebnisse je Aktivierungsfunktion, Schluessel "Sigmoid" bzw. "ReLU"
for act in (nn.Sigmoid, nn.ReLU):        # die Schleife laeuft ueber zwei Klassen (noch keine Objekte)
    torch.manual_seed(0)                 # beide Netze starten mit denselben Zufallszahlen
    # (100,) * 10 = (100, 100, ..., 100): Tupel mit 10 Eintraegen -> 10 verdeckte Schichten
    m = MLP(hidden=(100,) * 10, act=act, init="glorot")        # 10 verdeckte Schichten a 100, Glorot wie Abb. 17
    loss0, zn, wn = grad_norms_first_batch(m)   # die drei Rueckgabewerte auf drei Variablen verteilen
    deep[act.__name__] = {"model": m, "z_norms": zn, "w_norms": wn, "loss0": loss0}   # __name__ = Klassenname als Text
    # {:.2e} = wissenschaftliche Schreibweise, z. B. 3.21e-05 = 0,0000321
    # wn[0] ist die erste Linear-Schicht = model.net[1] (model.net[0] ist das Flatten)
    print(f"{act.__name__:8s}: model.net[1].weight.grad.norm() = {wn[0]:.2e} | "
          f"||dL/dz_1|| = {zn[0]:.2e} | ||dL/dz_10|| = {zn[-1]:.2e}")

# TODO: ||dL/dz_l|| ueber der Schichtnummer plotten (log-Skala!), beide Aktivierungen in einem Plot
#   Hinweis: Die Werte stehen in deep["Sigmoid"]["z_norms"] bzw. deep["ReLU"]["z_norms"];
#            Schichtnummern z. B. range(1, 11); log-Skala mit plt.yscale("log").


In [ ]:
# Lernt das tiefe Netz ueberhaupt? 3 Epochen auf einem 10 000er-Subset, SGD lr=0.1
sub10k = Subset(train, range(10_000))   # nur die ersten 10 000 Trainingsbilder (spart Rechenzeit)
dl10k = make_loaders(sub10k)            # Loader fuer dieses Subset; Val/Test bleiben vollstaendig
h_deep = {}
for name, d in deep.items():            # .items() liefert Paare (Schluessel, Wert), z. B. ("ReLU", {...})
    print(name)
    # TODO: SGD-Optimierer (lr=0.1) auf d["model"], 3 Epochen mit fit(..., loaders=dl10k)
    #   Hinweis: Der Optimierer braucht d["model"].parameters() -- analog zu Adam in Aufgabe 1.
    h_deep[name] = ...

# TODO: beide Verlaeufe plotten; Referenzlinie ln(10) einzeichnen (plt.axhline)
#   Hinweis: ln(10) = math.log(10) ist der Verlust bei reinem Raten -- bleibt eine Kurve dort, lernt das Netz nichts.


### Interpretation Aufgabe 2

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 3 · Optimierer × Lernrate

SGD, SGD+Momentum (0,9) und Adam mit Lernraten $\{10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}\}$, je 2 Epochen. **Rechenbudget:** Der Sweep läuft auf dem 10 000er-Subset `dl10k` aus Aufgabe 2 (12 Läufe × 2 Epochen ≈ 2–3 Min. auf 4 Kernen); validiert wird auf dem vollen Validierungssplit. Welche Kombination erreicht den geringsten Validierungsverlust, welche divergiert?

In [ ]:
# Dictionary: Name -> Funktion, die zu Parametern p und Lernrate lr einen neuen Optimierer erzeugt.
# "lambda p, lr: ..." ist eine Kurzschreibweise fuer eine kleine namenlose Funktion, gleichbedeutend mit
#     def make(p, lr): return torch.optim.SGD(p, lr=lr)
# So laesst sich unten fuer jeden Lauf ein frischer Optimierer bauen.
OPTIMIZERS = {
    "SGD":          lambda p, lr: torch.optim.SGD(p, lr=lr),
    "SGD+Momentum": lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9),
    "Adam":         lambda p, lr: torch.optim.Adam(p, lr=lr),
}
LRS = [1e-4, 1e-3, 1e-2, 1e-1]   # 0,0001 bis 0,1 -- jeweils Faktor 10

sweep = {}                       # Ergebnisse; Schluessel ist das Tupel (Optimierername, Lernrate)
for oname, make_opt in OPTIMIZERS.items():   # aeussere Schleife: 3 Optimierer
    for lr in LRS:                           # innere Schleife: 4 Lernraten -> zusammen 12 Laeufe
        torch.manual_seed(0)              # gleicher Startpunkt fuer alle Laeufe!
        m = MLP(hidden=(128, 128))        # je Lauf ein neues Netz -- sonst wuerde das vorige weitertrainiert
        # TODO: 2 Epochen auf dl10k trainieren (verbose=False), history in sweep[(oname, lr)] ablegen
        #   Hinweis: Optimierer erzeugen mit make_opt(m.parameters(), lr)
        ...
        # TODO: val loss nach 2 Epochen ausgeben; nicht endliche Werte (NaN/inf) als "divergiert" markieren
        #   Hinweis: letzter Wert einer Liste = liste[-1]; math.isfinite(x) ist False fuer NaN und inf

# TODO: Kombination mit dem geringsten Validierungsverlust bestimmen
#   Hinweis: divergierte Laeufe vorher ausschliessen -- Vergleiche mit NaN liefern immer False.


In [ ]:
# TODO: Validierungsverlust ueber den Epochen fuer alle 12 Kombinationen plotten
#       (ein Subplot je Optimierer, eine Linie je Lernrate, log-Skala) und als 3x4-Tabelle ausgeben
#   Hinweis: fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True) erzeugt drei Achsen nebeneinander;
#            axes[0], axes[1], axes[2] verwenden Sie dann wie ax in plot_history (ax.plot, ax.set_yscale, ...).
#            Die Tabelle genuegt als formatierte print-Ausgabe mit f-Strings (eine Zeile je Optimierer).


### Interpretation Aufgabe 3

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 4 · Regularisierungs-Ablation

Training auf nur 1 000 Beispielen (künstliche Datenknappheit), 50 Epochen, Adam $10^{-3}$. Vergleich: keine Regularisierung / Weight Decay $10^{-3}$ / Dropout 0,5 / Early Stopping (patience 5). Begründen Sie mit Tab. 4 des Skripts, welche Kombination Sie empfehlen. Die Testgenauigkeit wird je Variante genau einmal am Ende ermittelt.

In [ ]:
sub1k = Subset(train, range(1_000))                # kuenstliche Datenknappheit
dl1k = make_loaders(sub1k)                          # 16 Batches/Epoche, Val/Test bleiben voll

# Vier Varianten als Dictionary: Name -> Einstellungen (dict(a=1) bedeutet dasselbe wie {"a": 1}).
#   dropout       Anteil der Neuronen, die im Training zufaellig abgeschaltet werden (0.0 = aus)
#   weight_decay  Strafterm fuer grosse Gewichte; wird direkt dem Optimierer uebergeben (0.0 = aus)
#   patience      Early Stopping nach so vielen Epochen ohne Verbesserung (None = aus, volle 50 Epochen)
VARIANTS = {
    "keine Regularisierung":   dict(dropout=0.0, weight_decay=0.0,  patience=None),
    "Weight Decay 1e-3":       dict(dropout=0.0, weight_decay=1e-3, patience=None),
    "Dropout 0,5":             dict(dropout=0.5, weight_decay=0.0,  patience=None),
    "Early Stopping (pat. 5)": dict(dropout=0.0, weight_decay=0.0,  patience=5),
}

reg = {}
for name, cfg in VARIANTS.items():   # cfg ist das Einstellungs-Dictionary der aktuellen Variante
    torch.manual_seed(0)
    # TODO: MLP(hidden=(128,128), dropout=cfg["dropout"]) anlegen,
    #       Adam(lr=1e-3, weight_decay=cfg["weight_decay"]),
    #       50 Epochen auf dl1k mit patience=cfg["patience"] trainieren (verbose=False),
    #       anschliessend Test-Genauigkeit in h["test_acc"] ablegen und reg[name] = h
    #   Hinweis: Das Ergebnis von fit muss in der Variablen h landen -- die print-Zeile unten greift darauf zu.
    #            Einen neuen Eintrag in einem Dictionary legt man einfach per h["test_acc"] = ... an.
    ...
    # len(h['val_loss']) = Anzahl tatsaechlich trainierter Epochen (bei Early Stopping weniger als 50)
    print(f"{name}: Epochen {len(h['val_loss'])} | bestes val loss {min(h['val_loss']):.4f} | Test acc {h['test_acc']:.2%}")


In [ ]:
# TODO: Trainings- (hell) und Validierungsverlust (gestrichelt) aller Varianten in einem Plot,
#       Validierungsgenauigkeit in einem zweiten; Ergebnistabelle mit Epochenzahl, bestem val loss, Test-Genauigkeit
#   Hinweis: fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4)) erzeugt zwei Plots nebeneinander.
#            plot_history(reg, ax=ax1) ist ein moeglicher Ausgangspunkt; "hell" geht mit ax.plot(..., alpha=0.4).


### Interpretation Aufgabe 4

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Aufgabe 5 (optional) · Backpropagation in NumPy

Implementieren Sie den Rückwärtsdurchlauf für das einschichtige MLP aus Bsp. 4.1 des Skripts (ReLU + Softmax/Cross-Entropy) und verifizieren Sie die Gradienten per zentraler finiter Differenzen auf einem winzigen Netz in `float64`.

In [ ]:
def relu(z):  return np.maximum(z, 0)   # elementweise max(z, 0): negative Werte werden 0

def forward_np(params, X, y):
    """Einschichtiges MLP (Skript Bsp. 4.1): z = W x + b1, h = ReLU(z), a = V h + b2, L = mittlere CE.
    X: (N, D), y: (N,) int. Rueckgabe: Loss und Cache fuer den Rueckwaertsdurchlauf."""
    # params ist ein Dictionary mit vier Arrays. Formen (D Eingaenge, K verdeckte Neuronen, C Klassen):
    #   W (K, D), b1 (K,), V (C, K), b2 (C,)
    W, b1, V, b2 = params["W"], params["b1"], params["V"], params["b2"]
    # "@" ist die Matrixmultiplikation, ".T" transponiert. Jede ZEILE von X ist ein Beispiel, daher X @ W.T
    # statt W x. b1 (Form (K,)) wird automatisch zu jeder Zeile addiert ("Broadcasting").
    Z = X @ W.T + b1                     # (N, K)
    H = relu(Z)                          # (N, K)
    A = H @ V.T + b2                     # (N, C) Logits
    # e^a kann bei grossen Logits ueberlaufen. Das Zeilenmaximum abzuziehen aendert die Softmax mathematisch
    # nicht, haelt die Zahlen aber klein. keepdims=True behaelt die Form (N, 1), damit das Abziehen zeilenweise klappt.
    A_shift = A - A.max(1, keepdims=True)                      # numerisch stabile Softmax
    # log-Softmax: log p = a - log(Summe e^a); direkt im Log-Raum ist stabiler als erst p, dann log(p)
    logp = A_shift - np.log(np.exp(A_shift).sum(1, keepdims=True))
    # logp[np.arange(N), y] waehlt in Zeile i die Spalte y[i] = log-Wahrscheinlichkeit der richtigen Klasse
    loss = -logp[np.arange(len(y)), y].mean()
    # Zwischenergebnisse fuer backward_np aufheben ("Cache"), damit sie nicht neu berechnet werden muessen
    return loss, dict(X=X, Z=Z, H=H, P=np.exp(logp), y=y)

def backward_np(params, cache):
    """Adjungierte von hinten nach vorne (Skript Gl. 13-15, Bsp. 4.1). Beachten Sie die Mittelung ueber N."""
    X, Z, H, P, y = cache["X"], cache["Z"], cache["H"], cache["P"], cache["y"]
    N = len(y)
    # TODO: U2 = dL/da = (p_hat - y_onehot) / N                        (Gl. 14)
    # TODO: grads["V"] = U2^T H, grads["b2"] = Summe von U2 ueber N    (Gl. 15)
    # TODO: U1 = (U2 V) * 1[Z > 0]                                     (ReLU-VJP)
    # TODO: grads["W"] = U1^T X, grads["b1"] = Summe von U1 ueber N
    #   NumPy-Hinweise:
    #     One-Hot-Matrix:   Y = np.zeros_like(P); Y[np.arange(N), y] = 1
    #     Summe ueber N:    U.sum(0)  (Achse 0 = ueber alle Zeilen/Beispiele)
    #     Indikator 1[Z>0]: (Z > 0) ergibt True/False, das beim Multiplizieren als 1/0 zaehlt
    #     Transponieren:    U2.T ; Matrixprodukt: U2.T @ H
    grads = {}
    ...
    return grads

def finite_diff(params, X, y, eps=1e-5):
    """Zentrale Differenzen fuer jeden Parameter -- nur fuer winzige Netze bezahlbar."""
    # Prinzip: dL/dp ~ (L(p + eps) - L(p - eps)) / (2 eps). Jeder einzelne Parameterwert wird kurz verschoben,
    # der Verlust zweimal berechnet und der Wert danach zurueckgesetzt.
    num = {}
    for k, p in params.items():
        g = np.zeros_like(p)             # Array gleicher Form fuer die numerischen Gradienten
        for idx in np.ndindex(p.shape):  # alle Indizes von p durchlaufen, z. B. (0, 0), (0, 1), ...
            old = p[idx]
            # p ist dasselbe Array wie params[k] -> die Aenderung wirkt direkt im Netz
            p[idx] = old + eps; lp, _ = forward_np(params, X, y)
            p[idx] = old - eps; lm, _ = forward_np(params, X, y)
            p[idx] = old                 # Originalwert wiederherstellen
            g[idx] = (lp - lm) / (2 * eps)
        num[k] = g
    return num

# --- Testaufbau: winziges Netz mit Zufallswerten -------------------------------
rng = np.random.default_rng(0)           # Zufallsgenerator mit Seed 0
D, K, C, N = 5, 4, 3, 7                  # 5 Eingaenge, 4 verdeckte Neuronen, 3 Klassen, 7 Beispiele
# NumPy rechnet standardmaessig in float64 -- noetig, damit die finiten Differenzen genau genug sind
params = {"W": rng.normal(0, 1/np.sqrt(D), (K, D)), "b1": rng.normal(0, .1, K),
          "V": rng.normal(0, 1/np.sqrt(K), (C, K)), "b2": rng.normal(0, .1, C)}
X = rng.normal(size=(N, D)); y = rng.integers(0, C, N)   # zufaellige Eingaben und Labels 0..C-1

loss, cache = forward_np(params, X, y)
ana = backward_np(params, cache)         # analytische Gradienten (Ihre Implementierung)
num = finite_diff(params, X, y)          # numerische Gradienten (Referenz)
for k in params:
    # groesste Abweichung je Parameter -- stimmt Ihre Implementierung, liegt sie weit unter 1e-7
    print(f"  {k:3s}: max |analytisch - numerisch| = {np.abs(ana[k]-num[k]).max():.2e}   (Ziel: < 1e-7)")

### Interpretation Aufgabe 5

*Ihre Antwort (3–6 Sätze, mit Bezug auf das Skript):*

…

## Checkliste

- [ ] Notebook läuft mit *Kernel → Restart & Run All* fehlerfrei durch (oder: abgebrochene Läufe sind markiert und durch Referenzlogs ersetzt)
- [ ] Zu jeder Aufgabe: Kurven **und** Interpretation mit Skript-Bezug
- [ ] `results/` mit den JSON-Logs liegt bei (Seed, Konfiguration, Kurven – Abschn. 8: *Loggen Sie jeden Lauf*)